# Session 1: a robot arm is a chain of links and joints
**A. Real-world problem.** A pick-and-place robot lifts a block from one spot and sets it
down on another. Its motors only turn joints. Nobody tells the gripper "go to x, y": we
have to work out which joint angles put it there. Today: the forward half of that question.

**Plan.** (1) describe a 3-link arm by its link lengths and joint angles, (2) add angles
down the chain, (3) find the gripper, (4) find the longest reach and test a block.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
LINKS = np.array([0.30, 0.25, 0.15])      # link lengths in metres: shoulder-elbow, elbow-wrist, wrist-gripper
BLOCK = np.array([0.40, 0.30])            # the block we want to pick up (metres)

## Step 1: the angle of each link in the room
Each joint angle is measured *relative to the link before it*. So link 2 points in the
direction q1 + q2 and link 3 in the direction q1 + q2 + q3: angles add up down the chain.
These running totals are called the cumulative angles.

In [ ]:
def cumulative_angles(q):
    """Direction of every link measured from the x axis, in radians."""
    return np.cumsum(np.asarray(q, dtype=float))

## Step 2: walk along the chain
Start at the base (0, 0). Each link adds length * (cos angle, sin angle). The last point is
the gripper. The tool angle (which way the gripper points) is the sum of all joint angles.

In [ ]:
def link_points(q, links=LINKS):
    """Base, elbow, wrist, gripper as four (x, y) points in metres. Angles in radians."""
    c = cumulative_angles(q)
    steps = np.asarray(links)[:, None] * np.stack([np.cos(c), np.sin(c)], axis=1)
    return np.vstack([[0.0, 0.0], np.cumsum(steps, axis=0)])


def gripper(q, links=LINKS):
    return link_points(q, links)[-1]


def tool_angle(q):
    return float(np.sum(q))

## Step 3: how far can the arm reach, and can it touch the block?
The gripper can never be farther from the base than the sum of the link lengths (arm
straight). We can test a block by comparing distances.

In [ ]:
def max_reach(links=LINKS):
    return float(np.sum(links))


def touches(q, target, tol=0.02, links=LINKS):
    """True if the gripper is within tol metres of the target."""
    return bool(np.linalg.norm(gripper(q, links) - np.asarray(target)) <= tol)


def find_pose(target, tol=0.02, step_deg=5, links=LINKS):
    """Brute force: try joint angles q1, q2 in steps and q3 free; return the first pose that touches.
    (Slow and clumsy: Sessions 2 and 3 replace it with maths.)"""
    grid = np.radians(np.arange(-180, 180, step_deg))
    for a in grid:
        for b in grid:
            for c in grid:
                q = np.array([a, b, c])
                if touches(q, target, tol, links):
                    return q
    return None

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(5.8, 5.2))
    for q_d, col in [((0, 0, 0), "tab:gray"), ((30, 40, -20), "tab:blue"), ((70, -60, -30), "tab:green")]:
        q = np.radians(q_d)
        p = link_points(q)
        ax.plot(p[:, 0], p[:, 1], "o-", lw=4, color=col, label=f"q = {q_d} deg")
        print(f"q = {q_d} deg -> gripper {p[-1].round(3)} m, tool angle {np.degrees(tool_angle(q)):.0f} deg")
    ax.add_patch(plt.Circle((0, 0), max_reach(), fill=False, ls="--", color="gray"))
    ax.plot(*BLOCK, "s", ms=12, color="tab:red", label="block (0.40, 0.30) m")
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title("A 3-link arm in three poses; dashed: longest reach 0.70 m")
    fig.savefig(OUT / "arm_poses.png", dpi=120)

    q1 = np.radians(np.linspace(-180, 180, 361))
    xy = np.array([gripper([a, np.radians(40), np.radians(-20)]) for a in q1])
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(np.degrees(q1), xy[:, 0], label="gripper x")
    ax2.plot(np.degrees(q1), xy[:, 1], label="gripper y")
    ax2.set_xlabel("shoulder angle q1 (degrees), q2 = 40, q3 = -20")
    ax2.set_ylabel("gripper coordinate (m)")
    ax2.set_title("Turning only the shoulder swings the gripper round a circle")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "gripper_vs_q1.png", dpi=120)
    pose = find_pose(BLOCK, step_deg=10)
    print("brute-force pose for the block (degrees):", None if pose is None else np.degrees(pose).round(0))

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** Change the angles in `lesson.py` to (45, 45, 45) degrees. Predict the gripper position by hand first, then run it.
2. **Stretch.** Add a fourth link of 0.10 m to `LINKS` and a fourth angle, and predict the new longest reach before you run `max_reach`.
3. **Explain.** Explain why the pose (90, -90, 0) degrees touches the block, using the cumulative angles.

## Assessment
- **A. Conceptual.** Why is the tool angle q1 + q2 + q3, and not just q3? What would the gripper do if only joint 1 turned by 10 degrees?
- **B. Numerical.** For q = (90, -90, 0) degrees find the gripper position. Is it on the block at (0.40, 0.30) m?
- **C. Graph and simulation interpretation.** In the second figure, how large is the radius of the circle the gripper runs round, and at which shoulder angle is the gripper's x largest?
- **D. Design and optimisation challenge.** Design: a supplier sells 3-link arms with total link length 0.45, 0.60, 0.70 and 0.85 m. The block is 0.50 m from the base and you want at least 0.10 m of spare reach. Which arm do you choose, and what do you give up with a longer one?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    d = np.radians
    checks.append(("straight arm: gripper at (0.70, 0) m and longest reach 0.70 m",
                   np.allclose(L.gripper([0, 0, 0]), [0.70, 0.0], atol=1e-12) and abs(L.max_reach() - 0.70) < 1e-12))
    checks.append(("cumulative angles of (30, 40, -20) degrees are 30, 70, 50",
                   np.allclose(np.degrees(L.cumulative_angles(d([30, 40, -20]))), [30, 70, 50])))
    g = L.gripper(d([30, 40, -20]))
    checks.append(("worked example: gripper at (0.442, 0.500) m", np.allclose(g, [0.4417, 0.4998], atol=1e-3)))
    checks.append(("worked example: tool angle 50 degrees", abs(np.degrees(L.tool_angle(d([30, 40, -20]))) - 50) < 1e-9))
    checks.append(("pose (70, -60, -30) degrees puts the gripper at (0.490, 0.274) m",
                   np.allclose(L.gripper(d([70, -60, -30])), [0.490, 0.274], atol=1e-3)))
    hand = (0.30 * np.cos(d(30)) + 0.25 * np.cos(d(70)) + 0.15 * np.cos(d(50)),
            0.30 * np.sin(d(30)) + 0.25 * np.sin(d(70)) + 0.15 * np.sin(d(50)))
    checks.append(("hand calculation agrees with link_points", np.allclose(L.link_points(d([30, 40, -20]))[-1], hand)))
    ok = True
    for a in np.linspace(-3, 3, 7):
        for b in np.linspace(-3, 3, 7):
            for c in np.linspace(-3, 3, 7):
                p = L.link_points([a, b, c])
                ok &= np.allclose(np.linalg.norm(np.diff(p, axis=0), axis=1), L.LINKS, atol=1e-9)
                ok &= np.linalg.norm(p[-1]) <= L.max_reach() + 1e-9
    checks.append(("link lengths never change and the gripper never goes beyond 0.70 m", bool(ok)))
    checks.append(("pose (90, -90, 0) degrees touches the block (0.40, 0.30) exactly",
                   L.touches(d([90, -90, 0]), L.BLOCK, 1e-9) and np.allclose(L.gripper(d([90, -90, 0])), [0.40, 0.30])))
    checks.append(("the block is 0.50 m from the base, inside the 0.70 m reach",
                   abs(np.linalg.norm(L.BLOCK) - 0.5) < 1e-12 and np.linalg.norm(L.BLOCK) < L.max_reach()))
    far = np.array([0.75, 0.0])
    checks.append(("a block 0.75 m away cannot be touched (brute force finds nothing)", L.find_pose(far, step_deg=30) is None))
    pose = L.find_pose(L.BLOCK, step_deg=10)
    checks.append(("brute force at 10 degree steps finds (-10, 70, 40) degrees, 1.3 cm from the block",
                   pose is not None and np.allclose(np.degrees(pose), [-10, 70, 40])
                   and abs(np.linalg.norm(L.gripper(pose) - L.BLOCK) - 0.0134) < 1e-3))
    a = d(np.linspace(-180, 180, 3601))
    x = np.array([L.gripper([t, d(40), d(-20)])[0] for t in a])
    r = np.linalg.norm(L.gripper([0, d(40), d(-20)]))
    checks.append(("shoulder sweep with q2 = 40, q3 = -20: radius 0.667 m, largest x at q1 = -18.5 degrees",
                   abs(r - 0.6670) < 1e-3 and abs(x.max() - r) < 1e-6 and abs(np.degrees(a[x.argmax()]) + 18.5) < 0.05))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
